# `Dynamic AI Knowledge Assistant`

In [45]:
# ============================================================
# PHASE 3 – METADATA GENERATION
# STEP 1 – VERIFY PHASE 2 OUTPUT
# ============================================================

from pathlib import Path
import pandas as pd
import re

# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

project_root = None

for parent in [Path.cwd()] + list(Path.cwd().parents):

    candidate = (
        parent
        / "data"
        / "processed"
        / "cleaned"
    )

    if candidate.exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be detected. "
        "Expected data/processed/cleaned directory."
    )

# ------------------------------------------------------------
# Define project paths
# ------------------------------------------------------------

CLEANED_DIR = (
    project_root
    / "data"
    / "processed"
    / "cleaned"
)

METADATA_DIR = (
    project_root
    / "data"
    / "metadata"
)

# ------------------------------------------------------------
# Verify cleaned documents
# ------------------------------------------------------------

cleaned_files = sorted(
    CLEANED_DIR.glob("*.txt")
)

print("Project root     :", project_root)
print("Cleaned directory:", CLEANED_DIR)
print("Metadata directory:", METADATA_DIR)

print("\nCleaned documents:", len(cleaned_files))

if len(cleaned_files) != 128:
    raise ValueError(
        f"Expected 128 cleaned documents, "
        f"found {len(cleaned_files)}."
    )

print("\nPhase 2 corpus verification PASSED.")

Project root     : c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant
Cleaned directory: c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\cleaned
Metadata directory: c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\metadata

Cleaned documents: 128

Phase 2 corpus verification PASSED.


In [46]:
# ============================================================
# STEP 2 – LOAD SOURCE METADATA
# ============================================================

# Phase 1 metadata backup is the permanent source metadata.
SOURCE_METADATA_PATH = (
    METADATA_DIR / "metadata_phase1_backup.csv"
)

FINAL_METADATA_PATH = (
    METADATA_DIR / "metadata.csv"
)

# ------------------------------------------------------------
# Create the Phase 1 backup if it does not already exist
# ------------------------------------------------------------

if not SOURCE_METADATA_PATH.exists():

    original_metadata_path = (
        METADATA_DIR / "metadata.csv"
    )

    if not original_metadata_path.exists():
        raise FileNotFoundError(
            "Neither the Phase 1 metadata nor its backup "
            "could be found."
        )

    original_metadata = pd.read_csv(
        original_metadata_path
    )

    required_phase1_columns = [
        "article_id",
        "title",
        "url",
        "category",
        "file_name",
        "word_count",
        "character_count",
        "content_hash",
        "download_date",
        "status"
    ]

    missing_columns = [
        column
        for column in required_phase1_columns
        if column not in original_metadata.columns
    ]

    if missing_columns:
        raise ValueError(
            "The existing metadata.csv does not appear to "
            f"contain the Phase 1 metadata schema. "
            f"Missing columns: {missing_columns}"
        )

    original_metadata.to_csv(
        SOURCE_METADATA_PATH,
        index=False,
        encoding="utf-8"
    )

    print(
        "Phase 1 metadata backup created:"
    )
    print(SOURCE_METADATA_PATH)

# ------------------------------------------------------------
# Load permanent Phase 1 metadata
# ------------------------------------------------------------

metadata_df = pd.read_csv(
    SOURCE_METADATA_PATH
)

print(
    "Source metadata:",
    SOURCE_METADATA_PATH
)

print(
    "Metadata records:",
    len(metadata_df)
)

Source metadata: c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\metadata\metadata_phase1_backup.csv
Metadata records: 128


In [47]:
# ============================================================
# Inspect metadata structure
# ============================================================

print("Metadata columns:")
print(metadata_df.columns.tolist())

print("\nMetadata shape:")
print(metadata_df.shape)

display(
    metadata_df.iloc[0]
)


Metadata columns:
['article_id', 'title', 'url', 'category', 'file_name', 'word_count', 'character_count', 'content_hash', 'download_date', 'status']

Metadata shape:
(128, 10)


article_id                                          114d69cdc919c60f
title                                        Artificial intelligence
url                https://en.wikipedia.org/wiki/Artificial_intel...
category                                     Artificial Intelligence
file_name                                Artificial_intelligence.txt
word_count                                                     13223
character_count                                                87181
content_hash       114d69cdc919c60fb857d0925882788c1f9b523547e3a2...
download_date                             2026-08-12 21:03:09.965773
status                                                        cached
Name: 0, dtype: object

In [48]:
# ============================================================
# VALIDATE EXISTING METADATA
# ============================================================

required_source_columns = [
    "article_id",
    "title",
    "url",
    "category",
    "file_name",
    "download_date"
]

missing_columns = [
    column
    for column in required_source_columns
    if column not in metadata_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required metadata columns: {missing_columns}"
    )

print("Required metadata columns: PASSED")


# ------------------------------------------------------------
# Record count
# ------------------------------------------------------------

if len(metadata_df) != 128:
    raise ValueError(
        f"Expected 128 metadata records, "
        f"found {len(metadata_df)}."
    )

print("Metadata record count:", len(metadata_df))


# ------------------------------------------------------------
# Duplicate article IDs
# ------------------------------------------------------------

duplicate_article_ids = (
    metadata_df["article_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate article IDs:",
    duplicate_article_ids
)

if duplicate_article_ids != 0:
    raise ValueError(
        "Duplicate article IDs detected."
    )


# ------------------------------------------------------------
# Duplicate filenames
# ------------------------------------------------------------

duplicate_file_names = (
    metadata_df["file_name"]
    .duplicated()
    .sum()
)

print(
    "Duplicate filenames:",
    duplicate_file_names
)

if duplicate_file_names != 0:
    raise ValueError(
        "Duplicate filenames detected."
    )


# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

missing_values = (
    metadata_df[required_source_columns]
    .isna()
    .sum()
)

print("\nMissing values:")
display(missing_values)

if missing_values.sum() != 0:
    raise ValueError(
        "Missing values detected in required metadata fields."
    )


print("\nExisting metadata validation PASSED.")

Required metadata columns: PASSED
Metadata record count: 128
Duplicate article IDs: 0
Duplicate filenames: 0

Missing values:


article_id       0
title            0
url              0
category         0
file_name        0
download_date    0
dtype: int64


Existing metadata validation PASSED.


In [49]:
# ============================================================
# VERIFY METADATA ↔ CLEANED FILE MAPPING
# ============================================================

cleaned_file_names = {
    file.name
    for file in cleaned_files
}

metadata_file_names = set(
    metadata_df["file_name"]
)

missing_cleaned_files = (
    metadata_file_names
    - cleaned_file_names
)

unexpected_cleaned_files = (
    cleaned_file_names
    - metadata_file_names
)

print(
    "Metadata files without cleaned documents:",
    len(missing_cleaned_files)
)

print(
    "Cleaned documents without metadata:",
    len(unexpected_cleaned_files)
)

if missing_cleaned_files:
    print("\nMissing cleaned files:")
    for file_name in sorted(missing_cleaned_files):
        print(file_name)

if unexpected_cleaned_files:
    print("\nUnexpected cleaned files:")
    for file_name in sorted(unexpected_cleaned_files):
        print(file_name)

if missing_cleaned_files or unexpected_cleaned_files:
    raise ValueError(
        "Metadata-to-document mapping validation failed."
    )

print("\nMetadata ↔ cleaned-document mapping PASSED.")

Metadata files without cleaned documents: 0
Cleaned documents without metadata: 0

Metadata ↔ cleaned-document mapping PASSED.


In [50]:
# ============================================================
# CALCULATE STATISTICS FROM CLEANED DOCUMENTS
# ============================================================

document_statistics = []

for file_path in cleaned_files:

    text = file_path.read_text(
        encoding="utf-8"
    )

    word_count = len(
        re.findall(r"\S+", text)
    )

    character_count = len(text)

    document_statistics.append({
        "file_name": file_path.name,
        "word_count": word_count,
        "character_count": character_count
    })

cleaned_statistics_df = pd.DataFrame(
    document_statistics
)

print(
    "Statistics calculated for:",
    len(cleaned_statistics_df),
    "documents"
)

display(
    cleaned_statistics_df.head()
)

Statistics calculated for: 128 documents


,file_name,word_count,character_count
0,1.58-bit_large_language_model.txt,609,4272
1,A.I._Artificial_Intelligence.txt,4226,26063
2,Active_learning_(machine_learning).txt,1545,10291
3,Agent_harness.txt,751,5017
4,AI_art.txt,3305,22332


In [51]:
# ============================================================
# COMPARE RAW AND CLEANED STATISTICS
# ============================================================

statistics_comparison = metadata_df[
    [
        "file_name",
        "word_count",
        "character_count"
    ]
].merge(
    cleaned_statistics_df,
    on="file_name",
    suffixes=(
        "_raw",
        "_cleaned"
    )
)

statistics_comparison[
    "word_count_difference"
] = (
    statistics_comparison["word_count_cleaned"]
    - statistics_comparison["word_count_raw"]
)

statistics_comparison[
    "character_count_difference"
] = (
    statistics_comparison["character_count_cleaned"]
    - statistics_comparison["character_count_raw"]
)

print(
    "Documents with word-count differences:",
    (
        statistics_comparison[
            "word_count_difference"
        ] != 0
    ).sum()
)

print(
    "Documents with character-count differences:",
    (
        statistics_comparison[
            "character_count_difference"
        ] != 0
    ).sum()
)

Documents with word-count differences: 0
Documents with character-count differences: 95


In [52]:
# ============================================================
# GENERATE FINAL METADATA
# ============================================================

final_metadata = metadata_df[
    [
        "title",
        "url",
        "category",
        "file_name",
        "download_date"
    ]
].copy()

# Add statistics calculated from cleaned documents
final_metadata = final_metadata.merge(
    cleaned_statistics_df,
    on="file_name",
    how="left"
)

# Rename fields according to Phase 3 requirements
final_metadata = final_metadata.rename(
    columns={
        "title": "Article Name",
        "url": "URL",
        "category": "Category",
        "word_count": "Word Count",
        "character_count": "Character Count",
        "download_date": "Download Date"
    }
)

# Keep only the six required fields
final_metadata = final_metadata[
    [
        "Article Name",
        "URL",
        "Category",
        "Word Count",
        "Character Count",
        "Download Date"
    ]
]

print(
    "Final metadata shape:",
    final_metadata.shape
)

display(
    final_metadata.head()
)

Final metadata shape: (128, 6)


,Article Name,URL,Category,Word Count,Character Count,Download Date
0,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial_intel...,Artificial Intelligence,13223,87174,2026-08-12 21:03:09.965773
1,Artificial general intelligence,https://en.wikipedia.org/wiki/Artificial_gener...,Artificial Intelligence,5602,35819,2026-08-12 21:03:10.988638
2,A.I. Artificial Intelligence,https://en.wikipedia.org/wiki/A.I._Artificial_...,Artificial Intelligence,4226,26063,2026-08-12 21:03:12.008079
3,Timeline of artificial intelligence,https://en.wikipedia.org/wiki/Timeline_of_arti...,Artificial Intelligence,981,7231,2026-08-12 21:03:13.038940
4,History of artificial intelligence,https://en.wikipedia.org/wiki/History_of_artif...,Artificial Intelligence,14430,95762,2026-08-12 21:03:14.057315


In [53]:
# ============================================================
# FINAL METADATA VALIDATION
# ============================================================

expected_columns = [
    "Article Name",
    "URL",
    "Category",
    "Word Count",
    "Character Count",
    "Download Date"
]

# ------------------------------------------------------------
# Column validation
# ------------------------------------------------------------

assert final_metadata.columns.tolist() == expected_columns

print("Column structure: PASSED")


# ------------------------------------------------------------
# Record count
# ------------------------------------------------------------

assert len(final_metadata) == 128

print(
    "Record count:",
    len(final_metadata)
)


# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

missing_final = (
    final_metadata.isna().sum()
)

print("\nMissing values:")
display(missing_final)

assert missing_final.sum() == 0


# ------------------------------------------------------------
# Duplicate records
# ------------------------------------------------------------

duplicate_final = (
    final_metadata.duplicated().sum()
)

print(
    "Duplicate metadata records:",
    duplicate_final
)

assert duplicate_final == 0


# ------------------------------------------------------------
# Numeric validation
# ------------------------------------------------------------

assert (
    final_metadata["Word Count"] > 0
).all()

assert (
    final_metadata["Character Count"] > 0
).all()

print(
    "Positive word/character counts: PASSED"
)


# ------------------------------------------------------------
# URL validation
# ------------------------------------------------------------

invalid_urls = final_metadata[
    ~final_metadata["URL"]
    .astype(str)
    .str.startswith(
        "https://en.wikipedia.org/"
    )
]

print(
    "Invalid Wikipedia URLs:",
    len(invalid_urls)
)

assert len(invalid_urls) == 0


print("\nFinal metadata validation PASSED.")

Column structure: PASSED
Record count: 128

Missing values:


Article Name       0
URL                0
Category           0
Word Count         0
Character Count    0
Download Date      0
dtype: int64

Duplicate metadata records: 0
Positive word/character counts: PASSED
Invalid Wikipedia URLs: 0

Final metadata validation PASSED.


In [54]:
# ============================================================
# SAVE FINAL METADATA
# ============================================================

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

final_metadata_path = (
    METADATA_DIR / "metadata.csv"
)

final_metadata.to_csv(
    final_metadata_path,
    index=False,
    encoding="utf-8"
)

print("Final Phase 3 metadata saved:")
print(final_metadata_path)

Final Phase 3 metadata saved:
c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\metadata\metadata.csv


In [55]:
# ============================================================
# FINAL VALIDATION
# ============================================================

saved_metadata = pd.read_csv(
    final_metadata_path
)

print("=" * 70)
print("PHASE 3 – METADATA GENERATION VALIDATION")
print("=" * 70)

print(
    f"Metadata records       : {len(saved_metadata):,}"
)

print(
    f"Metadata columns       : {len(saved_metadata.columns):,}"
)

print(
    f"Total words            : "
    f"{saved_metadata['Word Count'].sum():,}"
)

print(
    f"Total characters       : "
    f"{saved_metadata['Character Count'].sum():,}"
)

print(
    f"Missing values         : "
    f"{saved_metadata.isna().sum().sum():,}"
)

print(
    f"Duplicate records      : "
    f"{saved_metadata.duplicated().sum():,}"
)

print(
    f"Unique articles        : "
    f"{saved_metadata['Article Name'].nunique():,}"
)

print(
    f"Unique URLs            : "
    f"{saved_metadata['URL'].nunique():,}"
)

print("=" * 70)

phase3_passed = (
    len(saved_metadata) == 128
    and saved_metadata.columns.tolist() == expected_columns
    and saved_metadata.isna().sum().sum() == 0
    and saved_metadata.duplicated().sum() == 0
    and saved_metadata["Article Name"].nunique() == 128
    and saved_metadata["URL"].nunique() == 128
    and (saved_metadata["Word Count"] > 0).all()
    and (saved_metadata["Character Count"] > 0).all()
)

if phase3_passed:
    print("PHASE 3 VALIDATION PASSED")
else:
    print("PHASE 3 VALIDATION FAILED")

print("=" * 70)

PHASE 3 – METADATA GENERATION VALIDATION
Metadata records       : 128
Metadata columns       : 6
Total words            : 357,876
Total characters       : 2,374,308
Missing values         : 0
Duplicate records      : 0
Unique articles        : 128
Unique URLs            : 128
PHASE 3 VALIDATION PASSED
